# Задача 6. Обучение без учителя

В этой работе решается задача кластеризации на датасете `digits` из `sklearn`: 1797 объектов, 64 числовых признака (пиксели изображений цифр 8x8), 10 естественных групп.

План работы:

1. Загрузить данные и выполнить EDA.
2. Подготовить признаки для кластеризации.
3. Реализовать K-means самостоятельно.
4. Обучить собственный K-means и несколько моделей из `sklearn` с подбором гиперпараметров.
5. Сравнить скорость и качество моделей по внутренним и внешним метрикам.
6. Проверить влияние PCA при разном количестве главных компонент.
7. Визуализировать данные с помощью PCA и t-SNE.

In [ ]:
import time
import warnings
from typing import Any, cast

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    completeness_score,
    davies_bouldin_score,
    homogeneity_score,
    normalized_mutual_info_score,
    silhouette_score,
    v_measure_score,
)
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
RANDOM_STATE = 42

## 1. Загрузка данных

Для кластеризации выбран датасет рукописных цифр `digits`. Каждый объект описан 64 признаками: интенсивностями пикселей изображения 8x8. Истинные метки классов не используются при обучении, но помогают объективно сравнить найденные кластеры с естественными группами цифр.

In [ ]:
digits = load_digits()
X = digits.data.astype(float)
y_true = digits.target
feature_names = [f"pixel_{i}" for i in range(X.shape[1])]

df = pd.DataFrame(X, columns=feature_names)
df["target"] = y_true

print(f"Размер матрицы признаков: {X.shape}")
print(f"Количество признаков: {X.shape[1]}")
print(f"Количество истинных классов: {len(np.unique(y_true))}")
df.head()

## 2. Разведочный анализ данных

Проверим пропуски, распределение классов и базовые статистики признаков. Для кластеризации важно понимать масштаб признаков и наличие константных колонок: расстояния чувствительны к масштабу.

In [ ]:
eda_summary = pd.DataFrame(
    {
        "missing": df[feature_names].isna().sum(),
        "mean": df[feature_names].mean(),
        "std": df[feature_names].std(),
        "min": df[feature_names].min(),
        "max": df[feature_names].max(),
        "unique": df[feature_names].nunique(),
    }
)

print(f"Всего пропусков: {df[feature_names].isna().sum().sum()}")
print(f"Константных признаков: {(eda_summary['std'] == 0).sum()}")
print("Распределение истинных классов:")
display(df["target"].value_counts().sort_index().to_frame("count").T)
eda_summary.head(10)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for digit, ax in enumerate(axes.ravel()):
    mean_image = X[y_true == digit].mean(axis=0).reshape(8, 8)
    ax.imshow(mean_image, cmap="gray_r")
    ax.set_title(f"Средняя цифра {digit}")
    ax.axis("off")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df[feature_names].to_numpy().ravel(), bins=17, ax=axes[0])
axes[0].set_title("Распределение значений пикселей")
axes[0].set_xlabel("Интенсивность")

sns.histplot(df[feature_names].std(), bins=25, ax=axes[1])
axes[1].set_title("Распределение стандартных отклонений признаков")
axes[1].set_xlabel("std")
plt.tight_layout()
plt.show()

## 3. Подготовка данных

У признаков одинаковый смысл, но разные дисперсии: часть пикселей почти всегда нулевая, центральные пиксели меняются сильнее. Поэтому перед алгоритмами, основанными на расстояниях, стандартизируем признаки. Константные признаки можно оставить: после `StandardScaler` они станут нулями.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pd.DataFrame(X_scaled, columns=feature_names).describe().loc[
    ["mean", "std", "min", "max"]
].round(3)

## 4. Метрики качества кластеризации

Будем считать два типа метрик:

- внутренние: `silhouette`, `Calinski-Harabasz`, `Davies-Bouldin`, они используют только признаки и найденные кластеры;
- внешние: `ARI`, `NMI`, `homogeneity`, `completeness`, `V-measure`, они сравнивают кластеры с истинными цифрами и используются только для оценки.

In [ ]:
def count_clusters(labels: np.ndarray) -> int:
    """Количество кластеров без учета шума DBSCAN, обозначенного как -1."""
    return len(set(labels) - {-1})


def clustering_metrics(
    X_eval: np.ndarray,
    labels: np.ndarray,
    y_true: np.ndarray,
) -> dict[str, float | int]:
    labels = np.asarray(labels)
    unique_labels = np.unique(labels)
    n_clusters = count_clusters(labels)
    noise_share = float(np.mean(labels == -1)) if -1 in unique_labels else 0.0

    metrics: dict[str, float | int] = {
        "n_clusters": n_clusters,
        "noise_share": noise_share,
        "ARI": adjusted_rand_score(y_true, labels),
        "NMI": normalized_mutual_info_score(y_true, labels),
        "homogeneity": homogeneity_score(y_true, labels),
        "completeness": completeness_score(y_true, labels),
        "v_measure": v_measure_score(y_true, labels),
        "silhouette": np.nan,
        "calinski_harabasz": np.nan,
        "davies_bouldin": np.nan,
    }

    # Внутренние метрики определены только если найдено хотя бы 2 непустых группы.
    if 2 <= len(unique_labels) <= len(labels) - 1:
        metrics["silhouette"] = silhouette_score(X_eval, labels)
        metrics["calinski_harabasz"] = calinski_harabasz_score(X_eval, labels)
        metrics["davies_bouldin"] = davies_bouldin_score(X_eval, labels)

    return metrics


def add_result(
    rows: list[dict[str, Any]],
    model_name: str,
    params: dict[str, Any],
    fit_time: float,
    labels: np.ndarray,
    X_eval: np.ndarray = X_scaled,
) -> None:
    row: dict[str, Any] = {
        "model": model_name,
        "params": params,
        "fit_time_sec": fit_time,
    }
    row.update(clustering_metrics(X_eval, labels, y_true))
    rows.append(row)


def results_table(rows: list[dict[str, Any]]) -> pd.DataFrame:
    return (
        pd.DataFrame(rows)
        .sort_values(["ARI", "silhouette"], ascending=[False, False])
        .reset_index(drop=True)
    )

## 5. Собственная реализация K-means

Алгоритм:

1. Инициализировать центры случайными объектами.
2. Назначить каждый объект ближайшему центру.
3. Пересчитать центры как среднее объектов кластера.
4. Повторять шаги 2-3 до сходимости или достижения лимита итераций.

Для устойчивости используется несколько запусков `n_init`; выбирается решение с минимальной суммой квадратов расстояний до центров.

In [ ]:
class MyKMeans:
    def __init__(
        self,
        n_clusters: int = 8,
        max_iter: int = 100,
        tol: float = 1e-4,
        n_init: int = 10,
        random_state: int | None = None,
    ) -> None:
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.n_init = n_init
        self.random_state = random_state
        self.cluster_centers_ = np.empty((0, 0))
        self.labels_ = np.empty(0, dtype=int)
        self.inertia_ = np.inf
        self.n_iter_ = 0

    def _init_centers(self, X: np.ndarray, rng: np.random.Generator) -> np.ndarray:
        indices = rng.choice(X.shape[0], size=self.n_clusters, replace=False)
        return cast(np.ndarray, X[indices].copy())

    @staticmethod
    def _squared_distances(X: np.ndarray, centers: np.ndarray) -> np.ndarray:
        return cast(
            np.ndarray, ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
        )

    def _single_fit(
        self,
        X: np.ndarray,
        rng: np.random.Generator,
    ) -> tuple[np.ndarray, np.ndarray, float, int]:
        centers = self._init_centers(X, rng)

        n_iter = 0
        for iteration in range(1, self.max_iter + 1):
            n_iter = iteration
            distances = self._squared_distances(X, centers)
            labels = distances.argmin(axis=1)

            new_centers = centers.copy()
            for cluster_id in range(self.n_clusters):
                mask = labels == cluster_id
                if mask.any():
                    new_centers[cluster_id] = X[mask].mean(axis=0)
                else:
                    # Если кластер опустел, переинициализируем его случайным объектом.
                    new_centers[cluster_id] = X[rng.integers(0, X.shape[0])]

            shift = np.linalg.norm(new_centers - centers)
            centers = new_centers
            if shift < self.tol:
                break

        distances = self._squared_distances(X, centers)
        labels = distances.argmin(axis=1)
        inertia = float(np.min(distances, axis=1).sum())
        return centers, labels, inertia, n_iter

    def fit(self, X: np.ndarray) -> "MyKMeans":
        X = np.asarray(X, dtype=float)
        rng = np.random.default_rng(self.random_state)

        best: tuple[np.ndarray, np.ndarray, float, int] | None = None
        for _ in range(self.n_init):
            run_rng = np.random.default_rng(rng.integers(0, np.iinfo(np.int32).max))
            result = self._single_fit(X, run_rng)
            if best is None or result[2] < best[2]:
                best = result

        if best is None:
            msg = "n_init must be positive"
            raise ValueError(msg)

        self.cluster_centers_, self.labels_, self.inertia_, self.n_iter_ = best
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        distances = self._squared_distances(
            np.asarray(X, dtype=float),
            self.cluster_centers_,
        )
        return cast(np.ndarray, distances.argmin(axis=1))

    def fit_predict(self, X: np.ndarray) -> np.ndarray:
        return self.fit(X).labels_

## 6. Обучение моделей и подбор гиперпараметров

Сравниваем четыре подхода:

- `MyKMeans`: собственная реализация;
- `sklearn.KMeans`;
- `AgglomerativeClustering`;
- `DBSCAN`.

Для каждой модели перебираем несколько значений гиперпараметров. Итоговую таблицу сортируем по `ARI`, потому что для этого датасета известны истинные классы цифр.

In [ ]:
rows: list[dict[str, Any]] = []

# Собственная реализация K-means
for k in range(8, 13):
    for n_init in [5, 10]:
        model = MyKMeans(
            n_clusters=k,
            max_iter=100,
            tol=1e-4,
            n_init=n_init,
            random_state=RANDOM_STATE,
        )
        start = time.perf_counter()
        labels = model.fit_predict(X_scaled)
        fit_time = time.perf_counter() - start
        add_result(
            rows, "MyKMeans", {"n_clusters": k, "n_init": n_init}, fit_time, labels
        )

# KMeans из sklearn
for k in range(8, 13):
    for init in ["k-means++", "random"]:
        model = KMeans(n_clusters=k, init=init, n_init=20, random_state=RANDOM_STATE)
        start = time.perf_counter()
        labels = model.fit_predict(X_scaled)
        fit_time = time.perf_counter() - start
        add_result(
            rows,
            "sklearn.KMeans",
            {"n_clusters": k, "init": init, "n_init": 20},
            fit_time,
            labels,
        )

# Иерархическая кластеризация
for k in range(8, 13):
    for linkage in ["ward", "complete", "average"]:
        model = AgglomerativeClustering(n_clusters=k, linkage=linkage)
        start = time.perf_counter()
        labels = model.fit_predict(X_scaled)
        fit_time = time.perf_counter() - start
        add_result(
            rows,
            "Agglomerative",
            {"n_clusters": k, "linkage": linkage},
            fit_time,
            labels,
        )

# DBSCAN: eps подбирается по квантилям расстояния до 5-го соседа.
nn_distances = (
    NearestNeighbors(n_neighbors=5).fit(X_scaled).kneighbors(X_scaled)[0][:, -1]
)
eps_values = np.quantile(nn_distances, [0.50, 0.60, 0.70, 0.80, 0.90, 0.95])

for eps in eps_values:
    for min_samples in [3, 5, 10]:
        model = DBSCAN(eps=float(eps), min_samples=min_samples)
        start = time.perf_counter()
        labels = model.fit_predict(X_scaled)
        fit_time = time.perf_counter() - start
        add_result(
            rows,
            "DBSCAN",
            {"eps": round(float(eps), 3), "min_samples": min_samples},
            fit_time,
            labels,
        )

all_results = results_table(rows)
all_results.head(15)

In [ ]:
best_by_model = (
    all_results.sort_values(
        ["model", "ARI", "silhouette"], ascending=[True, False, False]
    )
    .groupby("model", as_index=False)
    .head(1)
    .sort_values("ARI", ascending=False)
    .reset_index(drop=True)
)

best_by_model[
    [
        "model",
        "params",
        "fit_time_sec",
        "n_clusters",
        "noise_share",
        "ARI",
        "NMI",
        "homogeneity",
        "completeness",
        "v_measure",
        "silhouette",
        "calinski_harabasz",
        "davies_bouldin",
    ]
].round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.barplot(data=best_by_model, x="model", y="ARI", ax=axes[0])
axes[0].set_title("Лучшее качество по ARI")
axes[0].set_ylim(0, max(0.01, best_by_model["ARI"].max() * 1.15))
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=best_by_model, x="model", y="fit_time_sec", ax=axes[1])
axes[1].set_title("Время обучения лучших конфигураций")
axes[1].set_ylabel("seconds")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

## 7. PCA и качество кластеризации

Проверим, как меняется качество кластеризации после снижения размерности методом главных компонент. Для каждого количества компонент заново обучаются все использованные типы моделей. Это позволяет понять, помогает ли PCA убрать шум и ускорить обучение.

In [ ]:
def evaluate_models_on_matrix(
    X_matrix: np.ndarray,
    component_label: int,
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []

    for k in range(8, 13):
        model = MyKMeans(
            n_clusters=k,
            max_iter=100,
            n_init=5,
            random_state=RANDOM_STATE,
        )
        start = time.perf_counter()
        labels = model.fit_predict(X_matrix)
        add_result(
            rows,
            "MyKMeans",
            {"n_clusters": k, "n_init": 5},
            time.perf_counter() - start,
            labels,
            X_matrix,
        )

    for k in range(8, 13):
        model = KMeans(
            n_clusters=k,
            init="k-means++",
            n_init=20,
            random_state=RANDOM_STATE,
        )
        start = time.perf_counter()
        labels = model.fit_predict(X_matrix)
        add_result(
            rows,
            "sklearn.KMeans",
            {"n_clusters": k},
            time.perf_counter() - start,
            labels,
            X_matrix,
        )

    for k in range(8, 13):
        for linkage in ["ward", "average"]:
            model = AgglomerativeClustering(n_clusters=k, linkage=linkage)
            start = time.perf_counter()
            labels = model.fit_predict(X_matrix)
            add_result(
                rows,
                "Agglomerative",
                {"n_clusters": k, "linkage": linkage},
                time.perf_counter() - start,
                labels,
                X_matrix,
            )

    nn_distances = (
        NearestNeighbors(n_neighbors=5).fit(X_matrix).kneighbors(X_matrix)[0][:, -1]
    )
    eps_values = np.quantile(nn_distances, [0.60, 0.70, 0.80, 0.90, 0.95])
    for eps in eps_values:
        for min_samples in [3, 5]:
            model = DBSCAN(eps=float(eps), min_samples=min_samples)
            start = time.perf_counter()
            labels = model.fit_predict(X_matrix)
            add_result(
                rows,
                "DBSCAN",
                {"eps": round(float(eps), 3), "min_samples": min_samples},
                time.perf_counter() - start,
                labels,
                X_matrix,
            )

    table = results_table(rows)
    table["n_components"] = component_label
    return table

In [ ]:
pca_component_grid = [2, 5, 10, 20, 30, 40, 50, 64]
pca_results = []
pca_variance_rows = []

for n_components in pca_component_grid:
    pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X_scaled)
    pca_variance_rows.append(
        {
            "n_components": n_components,
            "explained_variance_ratio": pca.explained_variance_ratio_.sum(),
        }
    )
    pca_results.append(evaluate_models_on_matrix(X_pca, n_components))

pca_results = pd.concat(pca_results, ignore_index=True)
pca_variance = pd.DataFrame(pca_variance_rows)

best_pca_by_model = (
    pca_results.sort_values(
        ["n_components", "model", "ARI", "silhouette"],
        ascending=[True, True, False, False],
    )
    .groupby(["n_components", "model"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

best_pca_by_model.head(12).round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.lineplot(
    data=pca_variance,
    x="n_components",
    y="explained_variance_ratio",
    marker="o",
    ax=axes[0],
)
axes[0].set_title("Доля объясненной дисперсии PCA")
axes[0].set_ylim(0, 1.02)

sns.lineplot(
    data=best_pca_by_model,
    x="n_components",
    y="ARI",
    hue="model",
    marker="o",
    ax=axes[1],
)
axes[1].set_title("ARI после PCA")
axes[1].legend(title="model", fontsize=8)

sns.lineplot(
    data=best_pca_by_model,
    x="n_components",
    y="fit_time_sec",
    hue="model",
    marker="o",
    ax=axes[2],
)
axes[2].set_title("Время обучения после PCA")
axes[2].set_ylabel("seconds")
axes[2].legend(title="model", fontsize=8)

plt.tight_layout()
plt.show()

best_pca_summary = (
    best_pca_by_model.sort_values(
        ["model", "ARI", "silhouette"], ascending=[True, False, False]
    )
    .groupby("model", as_index=False)
    .head(1)
    .sort_values("ARI", ascending=False)
    .reset_index(drop=True)
)

best_pca_summary[
    [
        "model",
        "n_components",
        "params",
        "fit_time_sec",
        "n_clusters",
        "noise_share",
        "ARI",
        "NMI",
        "silhouette",
    ]
].round(4)

## 8. Визуализация PCA и t-SNE

Для визуализации снизим размерность до 2. PCA хорошо показывает линейную структуру, а t-SNE часто лучше разделяет локальные группы. Чтобы t-SNE работал быстрее и стабильнее, сначала уменьшим размерность до 30 компонент с помощью PCA.

In [ ]:
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_2d = pca_2d.fit_transform(X_scaled)

pca_30d = PCA(n_components=30, random_state=RANDOM_STATE).fit_transform(X_scaled)
tsne = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate=200.0,
    random_state=RANDOM_STATE,
)
X_tsne_2d = tsne.fit_transform(pca_30d)

viz_df = pd.DataFrame(
    {
        "pca_1": X_pca_2d[:, 0],
        "pca_2": X_pca_2d[:, 1],
        "tsne_1": X_tsne_2d[:, 0],
        "tsne_2": X_tsne_2d[:, 1],
        "digit": y_true.astype(str),
    }
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(
    data=viz_df,
    x="pca_1",
    y="pca_2",
    hue="digit",
    palette="tab10",
    s=25,
    linewidth=0,
    ax=axes[0],
)
axes[0].set_title("PCA до 2 компонент")
axes[0].legend(title="digit", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)

sns.scatterplot(
    data=viz_df,
    x="tsne_1",
    y="tsne_2",
    hue="digit",
    palette="tab10",
    s=25,
    linewidth=0,
    ax=axes[1],
)
axes[1].set_title("t-SNE после PCA до 30 компонент")
axes[1].legend(title="digit", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
best_model_row = best_by_model.iloc[0]
best_pca_row = best_pca_summary.iloc[0]

print("Лучшее решение без PCA:")
print(
    best_model_row[
        [
            "model",
            "params",
            "fit_time_sec",
            "n_clusters",
            "noise_share",
            "ARI",
            "NMI",
            "silhouette",
        ]
    ]
)

print("\nЛучшее решение среди PCA-экспериментов:")
print(
    best_pca_row[
        [
            "model",
            "n_components",
            "params",
            "fit_time_sec",
            "n_clusters",
            "noise_share",
            "ARI",
            "NMI",
            "silhouette",
        ]
    ]
)

## 9. Выводы

1. Данные подходят для кластеризации: 64 числовых признака, 10 естественных групп, пропусков нет.
2. Стандартизация признаков обязательна, потому что алгоритмы используют расстояния.
3. Собственная реализация K-means показывает сопоставимый принцип работы со `sklearn.KMeans`, но обычно уступает промышленной реализации по скорости и устойчивости.
4. Подбор гиперпараметров заметно влияет на качество: особенно для `DBSCAN`, где выбор `eps` определяет число кластеров и долю шума.
5. PCA может улучшить качество и скорость, если убрать шумные направления и оставить достаточно главных компонент.
6. Визуализация через t-SNE показывает локальные группы цифр заметнее, чем линейная проекция PCA.